In [7]:
import sys, os
sys.path.append(os.path.abspath("src"))  # add ./src to the Python path

In [4]:
!pip install python-dotenv

  Using cached python_dotenv-1.1.1-py3-none-any.whl.metadata (24 kB)
Using cached python_dotenv-1.1.1-py3-none-any.whl (20 kB)


In [9]:
import os, json
from datetime import datetime
import pandas as pd
from dotenv import load_dotenv
from storage import write_df, read_df

load_dotenv()
RAW = os.getenv("DATA_DIR_RAW", "data/raw")
PROC = os.getenv("DATA_DIR_PROCESSED", "data/processed")
os.makedirs(RAW, exist_ok=True)
os.makedirs(PROC, exist_ok=True)

RAW, PROC

('data/raw', 'data/processed')

In [11]:
import os
from glob import glob
from datetime import datetime
import pandas as pd

print("CWD:", os.getcwd())
print("RAW dir:", RAW)
print("Files in RAW now:", os.listdir(RAW))

# try several patterns
patterns = [
    os.path.join(RAW, "api_yahoo_AAPL_*.csv"),
    os.path.join(RAW, "api_*AAPL*.csv"),
    os.path.join(RAW, "*.csv"),
]

api_csv_in = None
for pat in patterns:
    matches = sorted(glob(pat))
    if matches:
        api_csv_in = matches[-1]  # latest match
        print("Found:", api_csv_in)
        break

# Fallback: recreate AAPL CSV if nothing matched
if api_csv_in is None:
    print("No Stage04 CSV found — creating one now with yfinance…")
    import yfinance as yf
    df_tmp = yf.download("AAPL", start="2024-01-01", end="2024-08-01")
    df_tmp.reset_index(inplace=True)
    ts = datetime.now().strftime("%Y%m%d-%H%M")
    api_csv_in = os.path.join(RAW, f"api_yahoo_AAPL_{ts}.csv")
    df_tmp.to_csv(api_csv_in, index=False)
    print("Saved:", api_csv_in)

# Load it
df = pd.read_csv(api_csv_in)
df.head()

CWD: /Users/yuri/bootcamp_Yuri_MoghaddamNasrollahi/homework/src
RAW dir: data/raw
Files in RAW now: []
No Stage04 CSV found — creating one now with yfinance…


/var/folders/zl/rp9hsgf15935fhdflxnc5q780000gn/T/ipykernel_96228/68364195.py:29: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_tmp = yf.download("AAPL", start="2024-01-01", end="2024-08-01")
[*********************100%***********************]  1 of 1 completed

Saved: data/raw/api_yahoo_AAPL_20250820-2355.csv


,Date,Close,High,Low,Open,Volume
0,NaN,AAPL,AAPL,AAPL,AAPL,AAPL
1,2024-01-02,184.0814971923828,186.85799339797134,182.34618895522584,185.5788148528252,82488700
2,2024-01-03,182.70318603515625,184.3195067154423,181.8900628291485,182.67343910134508,58414500
3,2024-01-04,180.38282775878906,181.552914127362,179.36147605378457,180.62080322049852,71983600
4,2024-01-05,179.65895080566406,181.22568820887324,178.6574354024576,180.46216340525032,62379700


In [12]:
from datetime import datetime

ts = datetime.now().strftime("%Y%m%d-%H%M")
raw_out = os.path.join(RAW, f"sample_{ts}.csv")
parq_out = os.path.join(PROC, f"sample_{ts}.parquet")

write_df(df, raw_out)
write_df(df, parq_out)

print("Saved:", raw_out, parq_out)

Saved: data/raw/sample_20250820-2355.csv data/processed/sample_20250820-2355.parquet


In [13]:
df_csv = read_df(raw_out)
df_parq = read_df(parq_out)

print("CSV shape:", df_csv.shape)
print("Parquet shape:", df_parq.shape)
print("Shapes match:", df_csv.shape == df_parq.shape)

print("Dtypes CSV:", df_csv.dtypes.astype(str).to_dict())
print("Dtypes Parquet:", df_parq.dtypes.astype(str).to_dict())

CSV shape: (147, 6)
Parquet shape: (147, 6)
Shapes match: True
Dtypes CSV: {'Date': 'object', 'Close': 'object', 'High': 'object', 'Low': 'object', 'Open': 'object', 'Volume': 'object'}
Dtypes Parquet: {'Date': 'object', 'Close': 'object', 'High': 'object', 'Low': 'object', 'Open': 'object', 'Volume': 'object'}


## Stage 05 — Data Storage

- Saved one dataset in **two formats**:
  - CSV → `data/raw/`
  - Parquet → `data/processed/`
- Reloaded both and validated shapes/dtypes match.
- Utilities (`write_df`, `read_df`) live in `src/storage.py`.
- Paths come from `.env` (`DATA_DIR_RAW`, `DATA_DIR_PROCESSED`).